# Cambios en percepción de inseguridad y victimización en el DMQ: comparación entre olas con emparejamiento por administración zonal

**Secretaría General de Seguridad Ciudadana y Gestión de Riesgos · Observatorio Metropolitano**

Este cuaderno reproduce el análisis de matching de la EVPI 2023-2024 vs 2025-2026 a nivel de Distrito, con control por administración zonal (AZ).

**Qué hace**
1. Carga la base EVPI (una fila por encuesta, ambas olas).
2. Construye los 56 indicadores del tablero territorial de percepción de inseguridad y victimización.
3. Empareja cada encuesta 2025-2026 con las 3 encuestas 2023-2024 más parecidas **dentro de su AZ** (sexo, edad, estudios, ingreso, NSE, trabaja).
4. Estima, por indicador, la diferencia entre olas (cruda, emparejada y ajustada por regresión con efectos fijos de AZ), con errores agrupados por barrio.
5. Corrige por comparaciones múltiples (Benjamini-Hochberg), clasifica con el criterio del tablero y exporta un Excel.

**Cómo usarlo:** ejecute las celdas en orden y cargue el archivo `.xlsx` de la EVPI cuando se le pida.

> Son comparaciones entre olas, no efectos causales; pueden reflejar cambios en el levantamiento además de cambios en las condiciones.

## 1. Parámetros y librerías

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# --- Parámetros del emparejamiento ---
K_VECINOS = 3                                   # controles de 2023-2024 por cada encuesta de 2025-2026
PESOS_DISTANCIA = {"ingreso": 3, "trabaja": 3, "edad_o": 2}  # peso extra en la distancia (resto = 1)
COL_PESO = "F_exp_personas"                     # factor de expansión de personas
COL_AZ = "ADMINISTRACION ZONAL"                 # variable de emparejamiento exacto y control
COL_CLUSTER = "dpa_barrio"                      # conglomerado para los errores estándar

## 2. Carga de la base
En Google Colab se abre el selector de archivos; fuera de Colab, defina la ruta en la variable de entorno `EVPI_XLSX` o en `RUTA`.

In [ ]:
RUTA = os.environ.get("EVPI_XLSX", "261002_EVPI_2024_2026_VF_1.xlsx")
if not os.path.exists(RUTA):
    try:
        from google.colab import files
        subido = files.upload()
        RUTA = list(subido.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"No se encontró {RUTA}. Defina la ruta en RUTA o en EVPI_XLSX.")

df = pd.read_excel(RUTA).reset_index(drop=True)
print("Encuestas por ola:", df["AÑO"].value_counts().sort_index().to_dict())

def col(codigo):
    """Devuelve el nombre completo de la columna a partir de su código (p. ej. 'PSC_1')."""
    return [c for c in df.columns if c.split(" - ")[0] == codigo][0]

## 3. Covariables socioeconómicas
Se usan para el emparejamiento y para el ajuste de la regresión. Los valores faltantes de estudios e ingreso se imputan con la mediana.

In [ ]:
df["w"] = df[COL_PESO].astype(float)
df["T"] = (df["AÑO"] == 2025).astype(float)             # 1 = ola 2025-2026 (grupo de interés); 0 = ola 2023-2024

df["mujer"] = (df[col("DD_1")] == "Mujer").astype(float)
df["edad_o"] = df["RANGO ETAREO"].map({"18 A 25": 1, "26 A 40": 2, "41 A 55": 3, "56 A 64": 4, "65 Y MÁS": 5}).astype(float)

orden_estudios = {"A": 0, "B": 2, "C": 1, "D": 4, "E": 3, "F": 6, "G": 5, "H": 8, "I": 7, "J": 9, "K": 9, "L": 9}
df["educ"] = df[col("DD_7")].str[0].map(orden_estudios).astype(float)
df["educ"] = df["educ"].fillna(df["educ"].median())

ing = df[col("DD_10")].str[0]
df["nsnc"] = (ing == "K").astype(float)                  # ingreso NS/NC
df["ingreso"] = ing.map({c: i for i, c in enumerate("ABCDEFGHIJ")}).astype(float)
df["ingreso"] = df["ingreso"].fillna(df["ingreso"].median())

df["trabaja"] = df[col("DD_8")].str.startswith("1").astype(float)
df["nse3f"] = df["nse3"].astype(float)

COVS = ["mujer", "edad_o", "educ", "ingreso", "nsnc", "nse3f", "trabaja"]
df[COVS].describe().round(2)

## 4. Indicadores del tablero
Los 56 indicadores y su dirección de mejora (`-1`: menos es mejor; `1`: más es mejor; `0`: sin dirección normativa). Se construyen en escala 0-100 (proporción de casos).

In [ ]:
# (id, bloque, nombre, dirección, base, advertencia)
INDICADORES = [
    ('PSC_1', 'A. Percepción de inseguridad', 'Inseguridad alrededor de la casa', -1, 'Excluye "No aplica"', ''),
    ('PSC_2', 'A. Percepción de inseguridad', 'Inseguridad en el barrio', -1, 'Excluye "No aplica"', ''),
    ('PSC_4', 'A. Percepción de inseguridad', 'Inseguridad cerca de centros educativos', -1, 'Excluye "No aplica"', ''),
    ('PSC_5', 'A. Percepción de inseguridad', 'Inseguridad en mercados y ferias', -1, 'Excluye "No aplica"', ''),
    ('PSC_6', 'A. Percepción de inseguridad', 'Inseguridad en cajeros de la vía pública', -1, 'Excluye "No aplica"', ''),
    ('PSC_7', 'A. Percepción de inseguridad', 'Inseguridad en espacios públicos de la ciudad', -1, 'Excluye "No aplica"', ''),
    ('PSC_8', 'A. Percepción de inseguridad', 'Inseguridad en el Metro de Quito', -1, 'Excluye "No aplica"', ''),
    ('PSC_9', 'A. Percepción de inseguridad', 'Inseguridad en trole, Ecovía y corredores', -1, 'Excluye "No aplica"', ''),
    ('PSC_11', 'A. Percepción de inseguridad', 'Inseguridad en taxis amarillos', -1, 'Excluye "No aplica"', ''),
    ('PSC_12', 'A. Percepción de inseguridad', 'Inseguridad en transporte por aplicaciones', -1, 'Excluye "No aplica"', ''),
    ('PSC_13', 'A. Percepción de inseguridad', 'Inseguridad en espectáculos públicos', -1, 'Excluye "No aplica"', ''),
    ('PSC_14', 'A. Percepción de inseguridad', 'Inseguridad al caminar solo/a por el barrio de día', -1, 'Excluye "No aplica"', ''),
    ('PSC_15', 'A. Percepción de inseguridad', 'Inseguridad al caminar solo/a por el barrio de noche', -1, 'Excluye "No aplica"', ''),
    ('PSC_16', 'B. Evolución percibida de la delincuencia', 'Percepción de que la delincuencia aumentó en el barrio', -1, 'Excluye "No aplica"', ''),
    ('PSC_17', 'B. Evolución percibida de la delincuencia', 'Percepción de que la delincuencia aumentó en Quito', -1, 'Excluye "No aplica"', ''),
    ('PSC_18', 'B. Evolución percibida de la delincuencia', 'Percepción de que la delincuencia aumentó en el país', -1, 'Excluye "No aplica"', ''),
    ('CC_1', 'C. Problemas percibidos en el barrio', 'Peleas o riñas en el barrio', -1, 'Excluye Ns/Nc', ''),
    ('CC_2', 'C. Problemas percibidos en el barrio', 'Pandillas o grupos delictivos en el barrio', -1, 'Excluye Ns/Nc', ''),
    ('CC_3', 'C. Problemas percibidos en el barrio', 'Consumo de alcohol en espacio público', -1, 'Excluye Ns/Nc', ''),
    ('CC_4', 'C. Problemas percibidos en el barrio', 'Venta clandestina de alcohol', -1, 'Excluye Ns/Nc', ''),
    ('CC_5', 'C. Problemas percibidos en el barrio', 'Consumo de drogas en espacio público', -1, 'Excluye Ns/Nc', ''),
    ('CC_6', 'C. Problemas percibidos en el barrio', 'Venta de drogas en espacio público', -1, 'Excluye Ns/Nc', ''),
    ('CC_7', 'C. Problemas percibidos en el barrio', 'Calles sin iluminación', -1, 'Excluye Ns/Nc', ''),
    ('CC_8', 'C. Problemas percibidos en el barrio', 'Apropiación de viviendas por grupos delictivos', -1, 'Excluye Ns/Nc', ''),
    ('CC_9', 'C. Problemas percibidos en el barrio', 'Reclutamiento de niñas, niños y adolescentes', -1, 'Excluye Ns/Nc', ''),
    ('TA_3', 'C. Problemas percibidos en el barrio', 'Reporte de civiles armados en el barrio', -1, 'Excluye Ns/Nc', ''),
    ('VI_PERS', 'D. Victimización (12 meses)', 'Victimización personal (al menos un delito)', -1, 'Todos los encuestados', ''),
    ('VI_HOG', 'D. Victimización (12 meses)', 'Victimización del hogar (al menos un delito)', -1, 'Todos los hogares; ítems filtrados = 0 si no aplica', ''),
    ('VI_6', 'D. Victimización (12 meses)', 'Robo con violencia o amenaza', -1, 'Todos los encuestados', ''),
    ('VI_7', 'D. Victimización (12 meses)', 'Hurto', -1, 'Todos los encuestados', ''),
    ('VI_8', 'D. Victimización (12 meses)', 'Estafa', -1, 'Todos los encuestados', ''),
    ('VI_9', 'D. Victimización (12 meses)', 'Fraude bancario', -1, 'Todos los encuestados', ''),
    ('VI_11', 'D. Victimización (12 meses)', 'Agresión física por desconocido', -1, 'Todos los encuestados', ''),
    ('VI_12', 'D. Victimización (12 meses)', 'Amenaza creíble', -1, 'Todos los encuestados', ''),
    ('VI_13', 'D. Victimización (12 meses)', 'Extorsión', -1, 'Todos los encuestados', ''),
    ('VI_10', 'D. Victimización (12 meses)', 'Pago de soborno a funcionario', -1, 'Todos los encuestados', ''),
    ('VI_4', 'D. Victimización (12 meses)', 'Robo a la vivienda', -1, 'Todos los hogares', ''),
    ('VI_14', 'D. Victimización (12 meses)', 'Secuestro', -1, 'Todos los hogares', ''),
    ('VI_1_1', 'D. Victimización (12 meses)', 'Robo de vehículo', -1, 'Solo hogares a los que aplica el filtro', ''),
    ('VI_2_1', 'D. Victimización (12 meses)', 'Robo de moto', -1, 'Solo hogares a los que aplica el filtro', ''),
    ('VI_3', 'D. Victimización (12 meses)', 'Robo de autopartes', -1, 'Solo hogares a los que aplica el filtro', ''),
    ('VI_5_1', 'D. Victimización (12 meses)', 'Robo a negocio en la vivienda', -1, 'Solo hogares a los que aplica el filtro', ''),
    ('CIE_4_1', 'E. Confianza institucional', 'Confianza en la Policía Nacional', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CIE_5_1', 'E. Confianza institucional', 'Confianza en las Fuerzas Armadas', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CIE_3_1', 'E. Confianza institucional', 'Confianza en la Fiscalía', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CIE_1_1', 'E. Confianza institucional', 'Confianza en juzgados y cortes', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CIE_2_1', 'E. Confianza institucional', 'Confianza en el ECU 911', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CIM_4_1', 'E. Confianza institucional', 'Confianza en los Agentes de Control Metropolitano', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CIM_5_1', 'E. Confianza institucional', 'Confianza en la Secretaría de Seguridad', 1, 'Solo quienes dicen conocer sus funciones', 'La proporción que conoce la institución cambió entre olas; base no plenamente comparable'),
    ('CH_1', 'F. Cambios de hábitos por inseguridad', 'Reducción de salidas nocturnas por inseguridad', -1, 'Excluye Ns/Nc y No aplica', ''),
    ('CH_2', 'F. Cambios de hábitos por inseguridad', 'Reducción de visitas a espacios recreativos por inseguridad', -1, 'Excluye Ns/Nc y No aplica', ''),
    ('CH_3', 'F. Cambios de hábitos por inseguridad', 'Restricción de salidas de hijos menores por inseguridad', -1, 'Excluye Ns/Nc y No aplica', ''),
    ('CH_4', 'F. Cambios de hábitos por inseguridad', 'Reducción de visitas a lugares turísticos por inseguridad', -1, 'Excluye Ns/Nc y No aplica', ''),
    ('CH_5', 'F. Cambios de hábitos por inseguridad', 'Reducción de asistencia a eventos por inseguridad', -1, 'Excluye Ns/Nc y No aplica', ''),
    ('CH_6', 'F. Cambios de hábitos por inseguridad', 'Reducción de caminatas por el barrio por inseguridad', -1, 'Excluye Ns/Nc y No aplica', 'En el DMQ, la respuesta "No aplica" cae de 17% a 2% entre olas; base no plenamente comparable'),
    ('SP_5', 'F. Cambios de hábitos por inseguridad', 'Participación en iniciativas barriales de seguridad', 0, 'Todos los encuestados', ''),
]

In [ ]:
def cat(codigo):
    return df[col(codigo)].astype("string")

def marca(codigo, si=("1",), no=("2",)):
    """1 si la respuesta empieza con un código de 'si', 0 si empieza con uno de 'no'; NaN en el resto (No aplica, Ns/Nc)."""
    s = cat(codigo)
    y = pd.Series(np.nan, index=df.index)
    y[s.str.startswith(tuple(no)).fillna(False)] = 0.0
    y[s.str.startswith(tuple(si)).fillna(False)] = 1.0
    return y

def victimizacion(codigo):
    """Pregunta de victimización con base 'todos': 1 si responde Sí, 0 en el resto."""
    s = cat(codigo)
    y = pd.Series(0.0, index=df.index)
    y[s.str.startswith("1").fillna(False)] = 1.0
    return y

Y = {}
for (i, bloque, nombre, direccion, base, adv) in INDICADORES:
    if i in ("PSC_16", "PSC_17", "PSC_18"):
        Y[i] = marca(i, si=("1",), no=("2", "3"))                 # "aumentó" vs. se mantiene/disminuyó
    elif i.startswith("PSC_") or i.startswith("CC_") or i == "TA_3" or i == "SP_5":
        Y[i] = marca(i)
    elif i.startswith("CH_"):                                      # reducción de actividades por inseguridad
        s = cat(i)
        y = pd.Series(np.nan, index=df.index)
        y[s.str.contains("Aument|mantuv", regex=True).fillna(False)] = 0.0
        y[s.str.contains("Disminuy").fillna(False)] = 1.0
        Y[i] = y
    elif i.startswith(("CIE_", "CIM_")) or i in ("VI_1_1", "VI_2_1", "VI_3", "VI_5_1"):
        Y[i] = marca(i)                                            # confianza (entre quienes conocen) e ítems filtrados
    elif i.startswith("VI_") and i not in ("VI_PERS", "VI_HOG"):
        Y[i] = victimizacion(i)

PERSONALES = ["VI_6", "VI_7", "VI_8", "VI_9", "VI_11", "VI_12", "VI_13"]
HOGAR = ["VI_1_1", "VI_2_1", "VI_3", "VI_4", "VI_5_1", "VI_14"]      # ítems filtrados = 0 si no aplica
Y["VI_PERS"] = pd.concat([Y[k].fillna(0) for k in PERSONALES], axis=1).max(axis=1)
Y["VI_HOG"] = pd.concat([Y[k].fillna(0) for k in HOGAR], axis=1).max(axis=1)
print(len(Y), "indicadores construidos")

## 5. Emparejamiento exacto por administración zonal
Para cada encuesta 2025-2026 se buscan, dentro de su AZ, las `K_VECINOS` encuestas de 2023-2024 con covariables más cercanas (distancia euclidiana estandarizada, con reemplazo). Cada control recibe el peso muestral de las encuestas a las que sirve de pareja, dividido por `K_VECINOS`.

In [ ]:
S = df.copy().reset_index(drop=True)
T = S["T"].values
w = S["w"].values

Z = ((S[COVS] - S[COVS].mean()) / S[COVS].std()).copy()
for v, peso in PESOS_DISTANCIA.items():
    Z[v] = Z[v] * np.sqrt(peso)
Z = Z.values

peso_control = np.zeros(len(S))
for az, idx in S.groupby(COL_AZ).groups.items():
    idx = np.array(list(idx))
    tratados = idx[T[idx] == 1]
    controles = idx[T[idx] == 0]
    D = ((Z[tratados][:, None, :] - Z[controles][None, :, :]) ** 2).sum(2)
    vecinos = np.argsort(D, axis=1)[:, :K_VECINOS]
    for a, i in enumerate(tratados):
        for j in vecinos[a]:
            peso_control[controles[j]] += w[i] / K_VECINOS

S["mw"] = np.where(T == 1, w, peso_control)     # pesos finales del emparejamiento
print("Controles únicos usados:", int((peso_control > 0).sum()), "de", int((T == 0).sum()))

## 6. Balance de covariables
Diferencia estandarizada (DEE) entre olas antes y después del emparejamiento.

In [ ]:
filas = []
for v in COVS:
    t = S[S["T"] == 1]; c = S[S["T"] == 0]
    sd = np.sqrt((t[v].var() + c[v].var()) / 2)
    m25 = np.average(t[v], weights=t["w"])
    m23 = np.average(c[v], weights=c["w"])
    m23e = np.average(c[v], weights=c["mw"])
    filas.append({"variable": v, "media_2025": m25, "media_2023_sin_matching": m23,
                  "media_2023_emparejado": m23e, "dee_antes": (m25 - m23) / sd, "dee_despues": (m25 - m23e) / sd})
BALANCE = pd.DataFrame(filas)
BALANCE.round(2)

## 7. Estimación por indicador
- **Diferencia cruda:** 2025-2026 menos 2023-2024 sin emparejar (pesos de expansión).
- **Diferencia emparejada:** misma diferencia sobre la muestra emparejada.
- **Diferencia ajustada (principal):** regresión ponderada sobre la muestra emparejada con las covariables y efectos fijos de AZ; errores agrupados por barrio.

Se reportan puntos porcentuales (pp).

In [ ]:
dummies_az = pd.get_dummies(S[COL_AZ], drop_first=True).astype(float).values
X_ajuste = np.column_stack([S[COVS].values.astype(float), dummies_az])
conglomerado = S[COL_CLUSTER].values

def estimar(y, pesos, ajustar):
    """MCO ponderado de y ~ T (+ covariables y AZ); EE agrupados por barrio (CR1). Devuelve coef., EE, p y medias."""
    m = ~np.isnan(y) & (pesos > 0)
    yy, ww, tt = y[m], pesos[m], T[m]
    X = np.column_stack([np.ones(m.sum()), tt] + ([X_ajuste[m]] if ajustar else []))
    XtW = X.T * ww
    A = XtW @ X
    b = np.linalg.solve(A, XtW @ yy)
    e = yy - X @ b
    grupos, inv = np.unique(conglomerado[m], return_inverse=True)
    G = np.zeros((len(grupos), X.shape[1]))
    np.add.at(G, inv, X * (ww * e)[:, None])
    n_g, n, k = len(grupos), len(yy), X.shape[1]
    Ai = np.linalg.inv(A)
    V = Ai @ (G.T @ G) @ Ai * n_g / (n_g - 1) * (n - 1) / (n - k)
    se = np.sqrt(V[1, 1])
    p = 2 * (1 - stats.t.cdf(abs(b[1] / se), n_g - 1))
    return {"coef": b[1], "se": se, "p": p,
            "media_t": np.average(yy[tt == 1], weights=ww[tt == 1]),
            "media_c": np.average(yy[tt == 0], weights=ww[tt == 0]),
            "n_t": int((tt == 1).sum()), "n_c": int((tt == 0).sum())}

filas = []
for (i, bloque, nombre, direccion, base, adv) in INDICADORES:
    y = Y[i].values.astype(float) * 100
    cruda = estimar(y, w, False)
    emp = estimar(y, S["mw"].values, False)
    aj = estimar(y, S["mw"].values, True)
    filas.append({"bloque": bloque, "id": i, "indicador": nombre, "dir": direccion, "base": base, "advertencia": adv,
                  "media_2023_sin_matching": cruda["media_c"], "media_2025": cruda["media_t"], "dif_cruda": cruda["coef"],
                  "media_2023_emparejado": aj["media_c"], "dif_emparejada": emp["coef"], "dif_ajustada": aj["coef"],
                  "ee": aj["se"], "ic95_inf": aj["coef"] - 1.96 * aj["se"], "ic95_sup": aj["coef"] + 1.96 * aj["se"],
                  "p": aj["p"], "n_2023": aj["n_c"], "n_2025": aj["n_t"]})
RES = pd.DataFrame(filas)

# --- Corrección por comparaciones múltiples (Benjamini-Hochberg) ---
p = RES["p"].values
orden = np.argsort(p)
q = np.empty(len(p)); previo = 1.0
for rango, i in enumerate(orden[::-1]):
    previo = min(previo, p[i] * len(p) / (len(p) - rango))
    q[i] = previo
RES["q_BH"] = q

# --- Clasificación con el criterio del tablero ---
def clasificar(r):
    if r["dir"] == 0:
        return "Sin dirección normativa" + (" (significativo)" if r["p"] < 0.05 else "")
    mejora = r["dif_ajustada"] * r["dir"] > 0
    if r["p"] < 0.05:
        return "Mejora significativa" if mejora else "Deterioro significativo"
    if r["p"] < 0.10:
        return "Señal de mejora por confirmar" if mejora else "Señal de deterioro por confirmar"
    return "Sin cambio significativo"
RES["clasificacion"] = RES.apply(clasificar, axis=1)
RES["clasificacion"].value_counts()

## 8. Resultados

In [ ]:
vista = RES[["id", "indicador", "media_2023_sin_matching", "media_2025", "dif_cruda", "dif_ajustada", "ic95_inf", "ic95_sup", "p", "q_BH", "clasificacion"]]
vista.round(2)

In [ ]:
resumen = (RES.assign(c=RES["clasificacion"].str.replace(r" \(significativo\)", "", regex=True))
              .pivot_table(index="bloque", columns="c", values="id", aggfunc="count", fill_value=0))
resumen

## 9. Exportar
Genera un Excel con los resultados por indicador y el balance.

In [ ]:
SALIDA = "resultados_matching_EVPI_DMQ_AZ.xlsx"
with pd.ExcelWriter(SALIDA, engine="openpyxl") as xl:
    RES.to_excel(xl, sheet_name="Indicadores_DMQ", index=False)
    BALANCE.to_excel(xl, sheet_name="Balance", index=False)
print("Guardado:", SALIDA)
try:
    from google.colab import files
    files.download(SALIDA)
except ImportError:
    pass